# Pre-trained CNNs on CIFAR — Evaluation, Freezing, Fine-tuning & Introspection
### Lecture notebook · VGG-16 · CIFAR-10 / CIFAR-100 · PyTorch + TorchVision

This notebook accompanies the computer-vision lecture on **transfer learning**. We take a CNN that was
pre-trained on ImageNet (VGG-16, 138 million parameters) and study, step by step, how it behaves on two
small datasets it has never seen. Everything is built from **modular, reusable functions and classes**, so
the same pipeline works with *any* TorchVision model and *any* image dataset — you only change two strings.

**Learning goals** — after working through this notebook you will be able to:

1. Explain what a **convolution operation** computes and how kernels filter an image.
2. Describe what **pre-training** means and why ImageNet features transfer to new tasks.
3. **Evaluate** a pre-trained model without any training (zero-shot) and interpret the (surprising) result.
4. **Freeze** layers of a network, train a new head, and reason about trainable-parameter counts.
5. **Progressively unfreeze** the last convolutional layers and measure how performance changes.
6. **Fully fine-tune** a pre-trained model with an appropriate learning rate.
7. **Visualize the internals** of a CNN: convolution kernels and feature (activation) maps, layer by layer.
8. Use a small, uniform experiment API (`init_experiment → train_experiment → visualize`) with any model/dataset.

---

## Table of contents

| § | Section | Content |
|---|---------|---------|
| 1 | [Setup & configuration](#1.-Setup-&-configuration) | imports, seeds, GPU check, global `CONFIG` |
| 2 | [Theory: the convolution operation](#2.-Theory:-the-convolution-operation) | worked demo with edge/blur kernels + quiz |
| 3 | [The modular toolkit](#3.-The-modular-toolkit) | data, models, freezing, training, visualization, unified API |
| 4 | [Part 1 — Pre-trained model as-is](#4.-Part-1-—-Pre-trained-model-as-is-(zero-shot)) | zero-shot evaluation on CIFAR-10/100 + linear probe |
| 5 | [Part 2 — Freezing layers](#5.-Part-2-—-Freezing-layers) | freeze backbone, train dense head |
| 6 | [Part 3 — Progressive unfreezing](#6.-Part-3-—-Progressive-unfreezing) | unfreeze last 1 / 2 / 3 conv layers |
| 7 | [Part 4 — Full fine-tuning](#7.-Part-4-—-Full-fine-tuning) | all parameters trainable |
| 8 | [Results summary](#8.-Results-summary) | comparison table + training curves |
| 9 | [Inside the model](#9.-Inside-the-model:-kernels,-feature-maps-&-t-SNE) | kernels, feature maps, t-SNE before/after |
| 10 | [Summary & further reading](#10.-Summary-&-further-reading) | take-aways, papers, links |

---

### How to run this notebook

* Recommended: **Google Colab** with a GPU — `Runtime → Change runtime type → T4 GPU`.
  The notebook auto-detects the device and falls back to CPU (slow but functional).
* **Run the cells top-to-bottom.** Later sections reuse variables (models, experiments) from earlier ones.
* Datasets (~170 MB each) and ImageNet weights (~528 MB) are downloaded once and cached.
* ⏱ **Expected runtimes (T4 GPU):** §4 ≈ 10 min · §5 ≈ 60–80 min · §6 ≈ 90–120 min · §7 ≈ 40–60 min.
* In a hurry? Set `CONFIG['speed_mode'] = True` in §1 — smaller images, fewer samples, same concepts.

> **Notation used below:** markdown cells with **✏️ Question** are short quizzes for the lecture.
> Think first — the answer is hidden under *Show answer*.

In [ ]:
# --- Optional: install dependencies (Google Colab already ships all of these) ---
# If you run the notebook on a bare machine, uncomment the next line:
# !pip install -q torch torchvision scikit-learn pandas matplotlib tqdm

import sys
print('Python:', sys.version.split()[0])
print('Everything needed for the lecture is importable — continue to Section 1.')

## 1. Setup & configuration

We import the libraries, fix the **random seeds** (so every run of the lecture produces the same numbers —
important for reproducible teaching), and select the **device**. PyTorch keeps a clear separation between
the model definition (pure Python) and the tensors that actually execute on hardware (`cpu` or `cuda`).
The next two cells must run before anything else, because every later cell uses `torch`, `device` and `CONFIG`.

In [ ]:
import random, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# ---- reproducibility: same shuffle order, same initial weights, same results ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- device: use the GPU if Colab provides one, otherwise CPU ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print('GPU found:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU. In Colab enable: Runtime > Change runtime type > T4 GPU')

print('torch', torch.__version__, '| torchvision', torchvision.__version__)

# nicer, larger default figures
plt.rcParams['figure.dpi'] = 110

In [ ]:
# --- Global configuration: ONE cell controls the whole lecture -------------------
# Change `model_name` / `dataset_name` and rerun any experiment — the modular
# toolkit (Section 3) accepts any TorchVision model and any supported dataset.

CONFIG = {
    # the pre-trained architecture to study in the lecture
    'model_name'  : 'vgg16',          # try: 'resnet18', 'resnet50', 'mobilenet_v2', ...
    # the target dataset for fine-tuning
    'dataset_name': 'CIFAR10',        # 'CIFAR10' or 'CIFAR100'
    # VGG-16 was trained on 224x224 ImageNet images -> we resize CIFAR up to 224.
    # Lower values (e.g. 96) run much faster; VGG's adaptive pooling absorbs any size.
    'image_size'  : 224,
    'batch_size'   : 32,              # images per optimisation step (GPU memory!)
    'num_workers'  : 2,               # background processes that load images

    # --- training budget per strategy (epochs) ---
    'epochs_frozen' : 5,              # Part 2: head training / frozen backbone
    'epochs_unfreeze': 4,             # Part 3: each progressive-unfreezing run
    'epochs_full'   : 5,              # Part 4: full fine-tuning

    # quick lecture mode: small images + data subsets (~20-30 min total)
    'speed_mode'   : False,
}

CONFIG

**What the configuration means.** `model_name` and `dataset_name` are the *only* two strings you must
change to retarget the whole notebook — every experiment in §§4–7 is written against the generic API.
`image_size = 224` matches the resolution VGG-16 saw during pre-training; feeding the network images at its
"native" resolution keeps the pre-trained statistics meaningful. `batch_size` trades GPU memory against
speed; 32 is safe for a free Colab T4. The three `epochs_*` entries define the training budget of each
experimental stage so that the full notebook fits into a single lab session. Finally, `speed_mode` shrinks
the images and subsamples the data when you want a complete run-through in under half an hour.

> 💡 **Teaching note:** the reason we can freely change `image_size` is that VGG-16 ends with an
> *adaptive* average pooling layer (`AdaptiveAvgPool2d((7,7))`), which converts *any* input spatial size
> into a fixed 7×7 grid before the classifier. We will meet this layer again in Section 3.

## 2. Theory: the convolution operation

Before touching the pre-trained model we refresh the operation that all of its 138 million parameters are
built from: **2-D convolution**. A convolution *slides* a small grid of numbers — a **kernel** (here 3×3) —
across the input image. At every position, each kernel entry multiplies the pixel underneath it, and the
nine products are summed into a **single output pixel**:

$$\text{out}(i, j) \;=\; \sum_{u,\,v} \text{kernel}(u, v)\,\cdot\,\text{in}(i+u,\; j+v)$$

Three properties matter for everything that follows:

* **The kernel is the learnable part.** Training a CNN does nothing more than adjust these small number
  grids. Early layers learn edge/color detectors, deeper layers combine them into textures, parts, objects.
* **Padding & stride.** With `padding=1` a 3×3 kernel gets a one-pixel border of zeros, so the output keeps
  the input's spatial size. A `stride` > 1 makes the kernel jump, shrinking the output — that is how
  pooling-like downsampling is built into convolutions.
* **Channels.** For RGB input the kernel is actually 3×3×**3** (one 3×3 grid per colour channel), and a
  convolution *layer* stacks many such filters — e.g. 64 — producing 64 output **feature maps**.

The demo below builds a synthetic image and convolves it with four *hand-crafted* kernels (a box blur, two
Sobel edge detectors, and a sharpening kernel). These are exactly the kind of filters that the first layer
of a trained CNN discovers **on its own** — a preview of Section 9, where we will look at the real, learned
kernels of VGG-16.

In [ ]:
# --- Worked demo: convolve a synthetic image with hand-crafted kernels -----------
torch.manual_seed(SEED)

# 1) a 64x64 grayscale image with stripes, a bright square and a disc
yy, xx = torch.meshgrid(torch.arange(64, dtype=torch.float32),
                        torch.arange(64, dtype=torch.float32), indexing='ij')
img = torch.zeros(1, 1, 64, 64)
img[:, :, :, ::8] = 0.35                                  # vertical stripes
img[:, :, 8:24, 8:24] = 1.00                              # bright square
img[:, :, (yy-44)**2 + (xx-44)**2 < 10**2] = 0.80         # disc (radius 10)

# 2) four classic hand-crafted 3x3 kernels (shape: [out_ch=1, in_ch=1, 3, 3])
kernels = {
    'box blur'  : torch.ones(1, 1, 3, 3) / 9.0,
    'sobel X'   : torch.tensor([[[[-1., 0., 1.],
                                  [-2., 0., 2.],
                                  [-1., 0., 1.]]]]),
    'sobel Y'   : torch.tensor([[[[-1., -2., -1.],
                                  [ 0.,  0.,  0.],
                                  [ 1.,  2.,  1.]]]]),
    'sharpen'   : torch.tensor([[[[ 0., -1.,  0.],
                                  [-1.,  5., -1.],
                                  [ 0., -1.,  0.]]]]),
}

# 3) convolve — F.conv2d(input, weight, padding=1) keeps the 64x64 size
outputs = {name: F.conv2d(img, k, padding=1) for name, k in kernels.items()}

# 4) draw: top row = the kernels themselves, bottom row = the filtered images
fig, axes = plt.subplots(2, 4, figsize=(11, 5.6))
for c, (name, k) in enumerate(kernels.items()):
    axes[0, c].imshow(k[0, 0], cmap='coolwarm', vmin=-3, vmax=3)
    axes[0, c].set_title(f'kernel\n"{name}"', fontsize=9)
    axes[1, c].imshow(outputs[name][0, 0], cmap='gray')
    axes[1, c].set_title('output', fontsize=9)
for ax in axes.ravel():
    ax.axis('off')
plt.suptitle('One convolution = slide kernel, multiply, sum -> new image', fontsize=11)
plt.tight_layout()
plt.show()

print('image shape :', tuple(img.shape), ' -> output shape:',
      tuple(outputs['sobel X'].shape))
print('Each kernel is a trainable 3x3 tensor — a CNN simply learns thousands of these.')

**Reading the demo.** The box blur averages each 3×3 neighbourhood — the output is a smoothed image.
The two Sobel kernels respond to *intensity gradients*: sobel-X fires on vertical edges (the stripe
boundaries and the square's left/right sides), sobel-Y on horizontal edges. The sharpen kernel amplifies
local contrast. Notice that every output keeps the full 64×64 size because we used `padding=1` — remove the
padding and the output would shrink to 62×62. Also note the shapes: the image is a 4-D tensor
`[batch=1, channels=1, height=64, width=64]` and each kernel is `[out_channels=1, in_channels=1, 3, 3]`.
A real convolution *layer* simply does this with, say, 64 different kernels *in parallel* and then passes
each result through a non-linearity such as ReLU.

> ✏️ **Question 2.1** — The Sobel-X kernel `[-1 0 1 / -2 0 2 / -1 0 1]` sums to zero. Why is a zero sum
> essential for an edge detector?
>
> ✏️ **Question 2.2** — An input is 32×32 with 3 channels and a layer has 64 kernels of size 3×3.
> How many numbers does one output feature map pixel aggregate, and how many parameters does the layer have?
>
> <details><summary><b>Show answers</b></summary>
>
> **2.1** A kernel whose weights sum to zero outputs ≈ 0 on flat regions (all equal values cancel) and
> only responds where intensity *changes* — i.e. edges. It also makes the detector invariant to overall
> brightness, because adding a constant to every pixel does not change the output.
>
> **2.2** Each output pixel aggregates 3 × 3 × 3 = **27 values** (kernel area × input channels).
> The layer has 64 kernels × 27 weights = **1,728 weights** (+64 biases). Note how small that is compared
> with a fully-connected layer on the same 32×32×3 input, which would need ~30 million weights — this
> *weight sharing* is the key efficiency of convolutions.
> </details>

### From convolution to pre-training

A CNN is nothing but a long chain of such convolutions (interleaved with non-linearities and pooling).
**Pre-training** means running this chain on a huge labelled dataset — historically
[ImageNet](https://www.image-net.org/): 1.28 million images, 1,000 everyday classes — with ordinary
supervised training. During those runs the kernels stop being random noise and organise themselves into a
hierarchy of *generic visual detectors*: edges in layer 1, corners and textures in layers 2–3, object parts
in layer 4, near-complete objects in layer 5.

The crucial empirical finding (and the reason this lecture exists) is that these detectors **transfer**:
a cat is a cat whether the photo comes from ImageNet or CIFAR. So instead of training a network from
scratch on our small 32×32 dataset, we *reuse* ImageNet's kernels and only adapt the last, task-specific
part. This notebook measures that idea quantitatively with four stages:

1. **Zero-shot** — use the pre-trained model exactly as it is (§4).
2. **Frozen** — keep the convolutional backbone fixed, train only the new classifier (§5).
3. **Progressive unfreezing** — release the last 1, 2, 3 convolutional layers one at a time (§6).
4. **Full fine-tuning** — train everything with a small learning rate (§7).

Each stage reuses the *same* modular code — that toolkit is up next.

## 3. The modular toolkit

This section builds the reusable engine of the lecture. Everything from here on is written against these
functions, which is what lets us run *any* model on *any* dataset with two-line experiment cells. The API
has four layers, each of which is one or two plain functions:

| Layer | Functions | Purpose |
|-------|-----------|---------|
| **Data** | `load_data(name, image_size, batch_size)` | CIFAR-10/100 with ImageNet-style preprocessing → loaders |
| **Model** | `build_model(name, num_classes, pretrained)` | any TorchVision CNN, with ImageNet weights |
| **Surgery** | `replace_head`, `freeze_all_conv_layers`, `unfreeze_last_convs`, `report_trainable` | adapt & control which weights train |
| **Experiment** | `init_experiment` → `train_experiment` → visualizers | uniform driver for all four lecture stages |

A first taste of the end result (this exact cell pattern returns in §§4–7):

```python
exp = init_experiment(model_name='vgg16', dataset_name='CIFAR10',
                      strategy='unfreeze3', epochs=4)
train_experiment(exp)        # trains, evaluates each epoch, logs into the results table
```

The `strategy` string is the *only* thing that changes between our four experimental stages:
`'zero_shot'`, `'head'`, `'frozen'`, `'unfreeze1'`, `'unfreeze2'`, `'unfreeze3'`, `'full'`.

### 3.1 Data — `load_data`

In [ ]:
# --- Data loading: ImageNet statistics + any CIFAR dataset ------------------------
# ImageNet pre-training fixes the *statistics* the network expects: every input
# must be normalised with ImageNet's per-channel mean and std, otherwise the
# pre-trained kernels receive numbers they were never calibrated for.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Build the preprocessing pipeline. split='train' adds a random flip (augmentation);
# split='test' is deterministic. Both resize the 32x32 CIFAR images to image_size.
def make_transforms(image_size, split):
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(),        # cheap, safe augmentation
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

# Download a torchvision dataset and build DataLoaders for it.
# Returns a dict with 'train'/'test' loaders, the class names and the class count.
# max_train / max_test cap the number of samples (used by speed_mode) -- this also
# makes the pipeline testable on a handful of images before a long training run.
def load_data(dataset_name='CIFAR10', image_size=224, batch_size=32,
              max_train=None, max_test=None, data_dir='./data'):
    registry = {'cifar10': (datasets.CIFAR10, 10),
                'cifar100': (datasets.CIFAR100, 100)}
    key = dataset_name.lower()
    if key not in registry:
        raise ValueError(f'Unknown dataset "{dataset_name}". Available: {list(registry)}')
    dataset_cls, num_classes = registry[key]

    train_ds = dataset_cls(data_dir, train=True,  download=True,
                           transform=make_transforms(image_size, 'train'))
    test_ds  = dataset_cls(data_dir, train=False, download=True,
                           transform=make_transforms(image_size, 'test'))

    if max_train:                                   # optional subsampling
        train_ds = Subset(train_ds, range(min(max_train, len(train_ds))))
    if max_test:
        test_ds = Subset(test_ds, range(min(max_test,  len(test_ds))))

    loaders = {
        'train': DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                            num_workers=CONFIG['num_workers'], pin_memory=True),
        'test' : DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                            num_workers=CONFIG['num_workers'], pin_memory=True),
        'classes': dataset_cls.classes,
        'num_classes': num_classes,
    }
    print(f'Dataset "{dataset_name}": {len(train_ds):,} train / {len(test_ds):,} test images, '
          f'{num_classes} classes, resized to {image_size}x{image_size}')
    return loaders

**What this cell does.** Two things deserve attention. First, the **normalisation constants**: TorchVision
releases ImageNet weights together with the exact mean/std preprocessing used during training — reusing
them verbatim is part of correct transfer learning, and mismatched statistics silently degrade accuracy.
Second, the **resize**: CIFAR images are 32×32, far smaller than the 224×224 the pre-trained kernels were
calibrated on, so we upscale. Up-scaling adds no information, but it preserves the *spatial scale* of the
features (a 3×3 kernel sees a correspondingly small patch of the image, just as it did on ImageNet).
The `max_train`/`max_test` arguments power `speed_mode` and let you smoke-test the pipeline on a handful
of images before committing to a long training run.

### 3.2 Model — `build_model`

`build_model` is deliberately generic: you hand it a **string**, it returns a ready-to-use CNN. The
`PRETRAINED_WEIGHTS` registry maps model names to TorchVision's ImageNet weight files; if a model is not
in the registry we still try to load its default pre-trained weights (`weights='DEFAULT'`). This is what
makes the notebook *model-agnostic*: swapping `'vgg16'` for `'resnet18'` or `'mobilenet_v2'` just works.

In [ ]:
# --- Model factory: any TorchVision CNN, optionally with ImageNet weights ---------
PRETRAINED_WEIGHTS = {
    'vgg16'           : models.VGG16_Weights.IMAGENET1K_V1,
    'resnet18'        : models.ResNet18_Weights.IMAGENET1K_V1,
    'resnet50'        : models.ResNet50_Weights.IMAGENET1K_V1,
    'mobilenet_v2'    : models.MobileNet_V2_Weights.IMAGENET1K_V1,
    'efficientnet_b0' : models.EfficientNet_B0_Weights.IMAGENET1K_V1,
}

# Create a TorchVision CNN by name.
#   pretrained=True  -> load ImageNet weights (the whole point of this lecture)
#   pretrained=False -> random initialisation (useful for fast parameter counting)
# The classifier head still outputs 1000 ImageNet classes here -- we replace it
# separately in replace_head() so that 'freezing' and 'replacing' are two
# independent, composable operations.
def build_model(model_name='vgg16', num_classes=10, pretrained=True):
    fn = getattr(models, model_name)                       # e.g. models.vgg16
    if pretrained:
        weights = PRETRAINED_WEIGHTS.get(model_name, 'DEFAULT')
        print(f'Building {model_name} with ImageNet pre-trained weights...')
    else:
        weights = None
        print(f'Building {model_name} with random weights (no pre-training)...')
    model = fn(weights=weights)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  -> {n_params/1e6:.1f}M parameters, head outputs 1000 ImageNet classes')
    return model

### 3.3 Head surgery — `replace_head`

The pre-trained head outputs **1000 ImageNet logits**; CIFAR-10 needs 10 and CIFAR-100 needs 100. We
therefore locate the network's **last `nn.Linear`** layer and swap it for a fresh one whose weights start
random. Two details make this generic rather than VGG-specific: we search `model.modules()` recursively
instead of hard-coding `model.classifier[6]`, and we keep the *old* input dimension (4096 for VGG-16,
512 for ResNet-18, 1280 for EfficientNet-B0 — whatever the backbone produces). An optional `hidden`
argument builds a small two-layer MLP head instead of a single linear layer — we will use that in Part 2
as the "brand-new head" strategy.

In [ ]:
# --- Head replacement: find the last Linear layer and swap it ----------------------

# Return (parent_module, attribute_name, linear_module) of the LAST nn.Linear
# found anywhere in the model. Works for VGG (classifier[6]), ResNet (fc),
# MobileNet/EfficientNet (classifier[-1]) -- i.e. any TorchVision CNN.
def _find_last_linear(model):
    found = (None, None, None)
    for module in model.modules():                 # depth-first, registration order
        for attr_name, child in module.named_children():
            if isinstance(child, nn.Linear):
                found = (module, attr_name, child) # keep overwriting -> last wins
    return found

# Swap the last Linear layer for a freshly initialised head.
#   hidden=None -> single Linear(old_in_features, num_classes)
#   hidden=int  -> small MLP: Linear(old, hidden) -> ReLU -> Dropout -> Linear(hidden, num_classes)
# The new layer's weights are random -- training them is what Parts 2-4 do.
def replace_head(model, num_classes, hidden=None):
    parent, attr_name, old_head = _find_last_linear(model)
    assert parent is not None, 'No Linear layer found in this model'
    if hidden is None:
        new_head = nn.Linear(old_head.in_features, num_classes)
    else:
        new_head = nn.Sequential(
            nn.Linear(old_head.in_features, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(hidden, num_classes),
        )
    setattr(parent, attr_name, new_head)
    print(f'Replaced head:  Linear({old_head.in_features} -> {old_head.out_features})'
          f'   ->   new head -> {num_classes} classes (randomly initialised)')
    return model

### 3.4 Freezing & unfreezing — controlling the gradient

Every PyTorch tensor has a flag `requires_grad`. If it is `False`, autograd never computes a gradient for
that tensor and the optimiser leaves it untouched — the layer is **frozen**: it still *runs* in the forward
pass (its outputs matter!) but it does not *learn*. Freezing is what makes transfer learning cheap: with
the backbone frozen we only back-propagate into the trainable part, and only those weights get updated.

The utilities below follow the same "find modules by type" pattern as `replace_head`, so they work on any
architecture. Note that `unfreeze_last_convs` collects conv layers **in definition order** and releases the
last *n* of them — for VGG-16, `n=3` unfreezes exactly the three 512→512 convolutions of block 5
(`features.24`, `features.26`, `features.28`), which is precisely experiment 3 of the lecture plan.

In [ ]:
# --- Freeze / unfreeze utilities (work for ANY TorchVision CNN) --------------------

# Freeze every parameter in the model.
def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

# Unfreeze every parameter (used by the 'full' strategy).
def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True

# Freeze only the convolutional layers (the visual 'backbone').
# Dense/classifier layers stay trainable. Returns how many conv layers were frozen.
def freeze_all_conv_layers(model):
    n_frozen = 0
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            for p in m.parameters():
                p.requires_grad = False
            n_frozen += 1
    print(f'Froze {n_frozen} convolutional layers (all conv weights + biases)')
    return n_frozen

# [(name, Conv2d), ...] for every conv layer, in definition order.
def get_conv_names(model):
    return [(name, m) for name, m in model.named_modules() if isinstance(m, nn.Conv2d)]

# Unfreeze the LAST n convolutional layers in definition order.
# VGG-16 with n=3 unfreezes features.24 / features.26 / features.28 (block 5).
def unfreeze_last_convs(model, n=1):
    convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    for m in convs[-n:]:
        for p in m.parameters():
            p.requires_grad = True
    print(f'Unfroze the last {n} of {len(convs)} conv layers:')
    for name, m in get_conv_names(model)[-n:][::-1]:
        print(f'   - {name}: Conv({m.in_channels} -> {m.out_channels}, '
              f'{m.kernel_size[0]}x{m.kernel_size[0]})')

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def total_params(model):
    return sum(p.numel() for p in model.parameters())

# Print trainable vs total parameters -- our main 'how much are we learning' meter.
def report_trainable(model, title='Model'):
    t, a = count_trainable(model), total_params(model)
    print(f'{title}: {t:,} trainable / {a:,} total parameters  ({100*t/a:.2f}%)')

In [ ]:
# --- Strategy applier: ONE function encodes the four lecture stages ----------------

STRATEGIES = ('zero_shot', 'head', 'frozen', 'unfreeze1', 'unfreeze2', 'unfreeze3', 'full')

STRATEGY_LABELS = {
    'zero_shot' : 'zero-shot (no training)',
    'head'      : 'new head only, everything frozen',
    'frozen'    : 'backbone frozen, dense layers trained',
    'unfreeze1' : 'backbone + last 1 conv unfrozen',
    'unfreeze2' : 'backbone + last 2 convs unfrozen',
    'unfreeze3' : 'backbone + last 3 convs unfrozen',
    'full'      : 'full fine-tuning (all trainable)',
}

# Configure `model` in-place for one of the lecture strategies.
#
# Order matters! We always FREEZE first and REPLACE THE HEAD last, because a
# newly created nn.Linear has requires_grad=True -- so the fresh head always
# ends up trainable, whatever was frozen before it.
#
#   zero_shot : random head for evaluation only, nothing is trained
#   head      : freeze EVERYTHING, then add a brand-new small MLP head
#   frozen    : freeze all conv layers, keep the (dense) classifier trainable
#   unfreezeN : 'frozen' + release the last N conv layers  (N = 1, 2, 3)
#   full      : everything trainable, head replaced for the new class count
def apply_strategy(model, strategy, num_classes):
    strategy = strategy.lower()
    if strategy == 'zero_shot':
        replace_head(model, num_classes)                  # random head, model untouched
    elif strategy == 'head':
        freeze_all(model)
        replace_head(model, num_classes, hidden=256)      # brand-new small head
    elif strategy == 'frozen':
        freeze_all_conv_layers(model)
        replace_head(model, num_classes)                  # dense layers remain trainable
    elif strategy in ('unfreeze1', 'unfreeze2', 'unfreeze3'):
        apply_strategy(model, 'frozen', num_classes)      # start from the frozen setup
        unfreeze_last_convs(model, int(strategy[-1]))     # ...then release last N convs
    elif strategy == 'full':
        unfreeze_all(model)
        replace_head(model, num_classes)
    else:
        raise ValueError(f'Unknown strategy "{strategy}". Choose from {STRATEGIES}')
    report_trainable(model, f'Strategy "{STRATEGY_LABELS[strategy]}"')
    return model

**Freezing — the mental model.** Think of the network as two parts: a *feature extractor* (all conv
layers, ≈14.7 M parameters in VGG-16) that turns pixels into a rich description, and a *classifier* (the
dense layers, ≈123 M parameters) that maps that description to labels. Under `strategy='frozen'` the
feature extractor keeps running exactly as ImageNet taught it, while the classifier — now ending in a
fresh 10-way layer — learns the new task. Under `'head'` even the old dense layers are frozen and only a
brand-new mini-head trains. And `unfreezeN` gradually hands the *visual* features themselves over to the
new task, starting from the most abstract (last) layer.

> ✏️ **Question 3.1** — After `apply_strategy(model, 'unfreeze3', 10)`, exactly which weight tensors still
> receive gradient updates? (List them by role, not by index.)
>
> ✏️ **Question 3.2** — Why do we call `freeze_*` *before* `replace_head` rather than the other way round?
>
> <details><summary><b>Show answers</b></summary>
>
> **3.1** (1) the whole dense classifier except the replaced last layer (classifier[0] and classifier[3]
> of VGG-16), (2) the *new* final Linear (4096→10), and (3) the weights+biases of the three last
> convolutions of block 5 (`features.24/26/28`). Everything earlier — conv blocks 1–4 — stays frozen.
>
> **3.2** Because a freshly created `nn.Linear` starts with `requires_grad=True`. If we replaced the head
> first and froze afterwards, `freeze_all` would freeze the *new* head too and nothing would train.
> Freezing first, replacing last, guarantees the new head is always trainable. A good habit: "freeze,
> then attach".
> </details>

### 3.5 Training & evaluation loops

The training loop is the classic five-beat pattern: *forward → loss → backward → step → repeat*, wrapped
with two production details worth teaching. **Mixed precision (AMP)**: on the GPU, activations and
gradients run in half precision (`float16`) where safe — roughly 2× faster and half the memory on Colab
T4s, with no code changes to the model itself. **`torch.no_grad()` evaluation**: during testing we skip
gradient bookkeeping entirely, which is faster and saves memory. Both functions return running
`(loss, accuracy)` so the caller only has to log them per epoch.

Notice that the optimiser receives **only the trainable parameters** — `(p for p in model.parameters() if
p.requires_grad)` — frozen tensors are not even handed to Adam. This single line is where the freeze
strategies of §§5–7 become real: whatever the strategy left trainable is exactly what gets updated.

In [ ]:
# --- One training epoch: forward -> loss -> backward -> step -----------------------
def train_one_epoch(model, loader, criterion, optimizer, device, epoch_idx=0):
    model.train()                                  # enable training behaviour (dropout!)
    running_loss, running_correct, seen = 0.0, 0, 0
    use_amp = (device.type == 'cuda')              # mixed precision only makes sense on GPU
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    for images, labels in tqdm(loader, desc=f'train ep{epoch_idx}', leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)      # 1) clear old gradients
        with torch.amp.autocast(device.type, enabled=use_amp):
            outputs = model(images)                # 2) forward pass
            loss = criterion(outputs, labels)      # 3) measure error
        scaler.scale(loss).backward()              # 4) back-propagate (scaled for fp16)
        scaler.step(optimizer)                     # 5) update trainable weights only
        scaler.update()

        running_loss   += loss.item() * labels.size(0)
        running_correct += (outputs.argmax(dim=1) == labels).sum().item()
        seen += labels.size(0)
    return running_loss / seen, running_correct / seen

# --- Evaluation: same forward pass, but no gradients and no weight updates ----------
@torch.no_grad()
def evaluate(model, loader, criterion, device, desc='eval'):
    model.eval()                                   # disable dropout etc.
    running_loss, running_correct, seen = 0.0, 0, 0
    use_amp = (device.type == 'cuda')
    for images, labels in tqdm(loader, desc=desc, leave=False):
        images, labels = images.to(device), labels.to(device)
        with torch.amp.autocast(device.type, enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss    += loss.item() * labels.size(0)
        running_correct += (outputs.argmax(dim=1) == labels).sum().item()
        seen += labels.size(0)
    return running_loss / seen, running_correct / seen

### 3.6 The `Experiment` container and the unified API

Everything the lecture does to a network is captured by five fields: **which model**, **which data**,
**which strategy**, **how many epochs**, and **which learning rate** — plus a `history` list that records
one dictionary per epoch. Grouping them in one small dataclass means every stage of §§4–7 is created,
trained, logged and plotted by the *same three functions*:

* **`init_experiment(...)`** — builds data + model, applies the freeze strategy, picks a sensible default
  learning rate per strategy, prints a parameter report, returns the `Experiment`.
* **`train_experiment(exp)`** — runs the epochs (or, for `zero_shot`, a single evaluation), records
  `history`, appends one row to the global `RESULTS` book, and plots the run's curves.
* **`results_table()`** — turns `RESULTS` into a pandas DataFrame for the §8 comparison.

The per-strategy default learning rates deserve a remark: the fresh head trains happily at `1e-3`, but
pre-trained conv weights are delicate — large updates would *destroy* the very features we want to keep,
so unfreezing stages drop to `3e-4` and full fine-tuning to `1e-4`. This "the deeper you unfreeze, the
gentler you step" rule is one of the lecture's key practical take-aways.

In [ ]:
# --- Experiment container + global results book ------------------------------------
from dataclasses import dataclass, field

@dataclass
class Experiment:
    name         : str          # human-readable label, e.g. 'vgg16 | CIFAR10 | frozen'
    model        : nn.Module    # the network, already strategy-configured & on device
    data         : dict         # loaders + class names + num_classes (from load_data)
    model_name   : str
    dataset_name : str
    strategy     : str          # one of STRATEGIES
    epochs       : int          # 0 for zero-shot (evaluation only)
    lr           : float        # learning rate chosen for this strategy
    history      : list = field(default_factory=list)   # one dict per epoch

RESULTS = []       # every train_experiment() call appends one summary row here

# Default learning rate per strategy: fresh heads like big steps,
# pre-trained weights need gentle ones.
LR_DEFAULTS = {
    'zero_shot': 0.0,
    'head'     : 1e-3,
    'frozen'   : 1e-3,
    'unfreeze1': 3e-4,
    'unfreeze2': 3e-4,
    'unfreeze3': 3e-4,
    'full'     : 1e-4,
}

# Record one experiment's outcome in the RESULTS book and print a summary line.
def log_results(exp):
    done = [e for e in exp.history if e.get('test_acc') is not None]
    best  = max((e['test_acc'] for e in done), default=float('nan'))
    last  = done[-1]['test_acc'] if done else float('nan')
    RESULTS.append({
        'experiment'   : exp.name,
        'model'        : exp.model_name,
        'dataset'      : exp.dataset_name,
        'strategy'     : STRATEGY_LABELS[exp.strategy],
        'trainable_M'  : round(count_trainable(exp.model) / 1e6, 2),
        'epochs'       : exp.epochs,
        'lr'           : exp.lr,
        'best_test_acc': round(best, 4),
        'final_test_acc': round(last, 4),
    })
    print(f'Logged: {exp.name} | best test acc so far: {best:.4f}')

# The results book as a pandas DataFrame (used again in Section 8).
def results_table():
    return pd.DataFrame(RESULTS)

In [ ]:
# --- init_experiment: build data + model + strategy in ONE call --------------------

def init_experiment(model_name=None, dataset_name=None, strategy='frozen',
                    epochs=None, lr=None, image_size=None, batch_size=None,
                    max_train=None, max_test=None, pretrained=True):
    # Create a fully configured Experiment. Any argument left as None falls back
    # to the global CONFIG cell from Section 1 -- so the lecture only ever needs
    # to spell out what it wants to change.
    model_name   = model_name   or CONFIG['model_name']
    dataset_name = dataset_name or CONFIG['dataset_name']
    strategy     = strategy.lower()
    if strategy not in STRATEGIES:
        raise ValueError(f'Unknown strategy "{strategy}". Choose from {STRATEGIES}')

    # speed_mode: trade resolution & data size for wall-clock time
    if CONFIG['speed_mode']:
        image_size = image_size or 96
        max_train  = max_train or 10000
        max_test   = max_test or 2000
    image_size = image_size or CONFIG['image_size']
    batch_size = batch_size or CONFIG['batch_size']

    # 1) data  2) model  3) strategy (freeze/unfreeze + head replacement)
    data  = load_data(dataset_name, image_size, batch_size, max_train, max_test)
    model = build_model(model_name, data['num_classes'], pretrained).to(device)
    apply_strategy(model, strategy, data['num_classes'])

    # 4) budget: per-strategy defaults from CONFIG
    if epochs is None:
        if   strategy in ('head', 'frozen'):            epochs = CONFIG['epochs_frozen']
        elif strategy.startswith('unfreeze'):           epochs = CONFIG['epochs_unfreeze']
        elif strategy == 'full':                        epochs = CONFIG['epochs_full']
        else:                                           epochs = 0    # zero-shot
    lr = lr if lr is not None else LR_DEFAULTS[strategy]

    exp = Experiment(
        name=f'{model_name} | {dataset_name} | {STRATEGY_LABELS[strategy]}',
        model=model, data=data, model_name=model_name,
        dataset_name=dataset_name, strategy=strategy, epochs=epochs, lr=lr)
    print(f'\nExperiment ready: "{exp.name}"  |  {epochs} epochs @ lr={lr}')
    return exp

In [ ]:
# --- train_experiment: run it, log it, plot it -------------------------------------

def plot_history(history, title=''):
    # Loss + accuracy curves for a single experiment.
    done = [e for e in history if e.get('train_loss') is not None]
    if not done:
        print('(no training epochs to plot — zero-shot run)')
        return
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    ep = [e['epoch'] for e in done]
    axes[0].plot(ep, [e['train_loss'] for e in done], 'o-', label='train')
    axes[0].plot(ep, [e['test_loss']  for e in done], 's-', label='test')
    axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].set_title('Loss')
    axes[1].plot(ep, [e['train_acc'] for e in done], 'o-', label='train')
    axes[1].plot(ep, [e['test_acc']  for e in done], 's-', label='test')
    axes[1].set_xlabel('epoch'); axes[1].set_ylabel('accuracy'); axes[1].set_title('Accuracy')
    for ax in axes: ax.grid(alpha=0.3); ax.legend()
    fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()

def train_experiment(exp, plot=True):
    # The single entry point used by every lecture stage.
    # For strategy='zero_shot' it only evaluates; otherwise it trains, evaluates
    # every epoch, records the history and appends a row to RESULTS.
    criterion = nn.CrossEntropyLoss()

    if exp.strategy == 'zero_shot':
        print('Zero-shot: evaluating the pre-trained model AS-IS (no training)...')
        loss, acc = evaluate(exp.model, exp.data['test'], criterion, device,
                             desc='zero-shot eval')
        exp.history.append({'epoch': 0, 'train_loss': None, 'train_acc': None,
                            'test_loss': loss, 'test_acc': acc})
        print(f'Zero-shot test accuracy: {acc:.4f}  ({acc*100:.1f}%)')
        log_results(exp)
        return exp

    # only trainable parameters are handed to the optimiser (frozen ones can't move)
    optimizer = torch.optim.Adam(
        (p for p in exp.model.parameters() if p.requires_grad), lr=exp.lr)
    print(f'Training "{exp.name}" for {exp.epochs} epochs @ lr={exp.lr} ...')

    for epoch in range(1, exp.epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(exp.model, exp.data['train'],
                                          criterion, optimizer, device, epoch)
        te_loss, te_acc = evaluate(exp.model, exp.data['test'],
                                   criterion, device, desc=f'test ep{epoch}')
        exp.history.append({'epoch': epoch, 'train_loss': tr_loss, 'train_acc': tr_acc,
                            'test_loss': te_loss, 'test_acc': te_acc,
                            'time_s': round(time.time() - t0, 1)})
        print(f'epoch {epoch:>2}/{exp.epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} '
              f'| test loss {te_loss:.4f} acc {te_acc:.4f} | {time.time()-t0:5.1f}s')

    log_results(exp)
    if plot:
        plot_history(exp.history, title=exp.name)
    return exp

### 3.7 Visualization functions

Three families of pictures, all reusable on any model from the toolkit:

* **`show_kernels`** — draws the *weights* of chosen conv layers: for the first layer each filter is a
  small RGB patch (3 input channels); deeper filters have many input channels, so we render their first
  channel as a grayscale patch. This is the "what did the network learn to look for?" view.
* **`show_feature_maps`** — runs one image through the network with **forward hooks** attached to chosen
  conv layers and draws the resulting activations: the "what does each layer actually see?" view.
* **`show_tsne`** — collects the feature vector just before the classifier for many test images and
  projects 25,088 dimensions down to 2-D; the "did fine-tuning reorganise the feature space?" view.

Plus two small helpers: `plot_histories` overlays accuracy curves of *several* experiments (used in §§6–8),
and `plot_trainable_params` visualises how the trainable count explodes across strategies (used in §5).

In [ ]:
# --- Kernel visualization: draw the learned filters of chosen conv layers ----------

# Undo the ImageNet normalisation so images are displayable again.
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD ).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

# Draw up to `max_filters` filters of each named conv layer, one row per layer.
# First-layer filters have 3 input channels -> shown as RGB patches.
# Deeper filters have many input channels -> we show their FIRST channel and
# renormalise each patch to [0, 1] for contrast.
def show_kernels(model, layer_names, max_filters=16, panel=1.1):
    layers = dict(get_conv_names(model))
    missing = [n for n in layer_names if n not in layers]
    assert not missing, f'Layers not found: {missing}. Use names from get_conv_names(model).'
    n_rows, n_cols = len(layer_names), max_filters
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols*panel, n_rows*(panel+0.35)))
    axes = np.atleast_2d(axes)
    for r, lname in enumerate(layer_names):
        w = layers[lname].weight.detach().cpu()
        n_show = min(max_filters, w.shape[0])
        for c in range(n_cols):
            ax = axes[r, c]; ax.axis('off')
            if c >= n_show: continue
            f = w[c]                                    # [in_ch, kH, kW]
            if f.shape[0] == 3:
                img = f.permute(1, 2, 0)                # RGB patch
                ax.imshow((img - img.min()) / (img.max() - img.min() + 1e-8))
            else:
                img = f[0]                              # first input channel
                ax.imshow(img, cmap='viridis',
                          vmin=img.min(), vmax=img.max())
        axes[r, 0].set_title(lname, loc='left', fontsize=8, pad=2)
    fig.suptitle('Learned convolution kernels (filters) — one row per layer',
                 fontsize=11, y=1.02)
    plt.tight_layout(); plt.show()

In [ ]:
# --- Feature-map visualization: what each layer activates on ONE image -------------
# Forward hooks capture a module's OUTPUT during a normal forward pass.
# We register one hook per requested layer, run the image once, remove the hooks.

def show_feature_maps(model, image, layer_names, max_maps=16, panel=1.0):
    layers = dict(get_conv_names(model))
    acts = {}
    handles = []
    for lname in layer_names:
        def make_hook(name):
            def hook(module, inputs, output):
                acts[name] = output.detach().cpu()
            return hook
        handles.append(layers[lname].register_forward_hook(make_hook(lname)))

    model.eval()
    with torch.no_grad():
        model(image.unsqueeze(0).to(next(model.parameters()).device))
    for h in handles:
        h.remove()                                     # always clean up hooks!

    n_rows, n_cols = len(layer_names), max_maps + 1    # +1 column for the input image
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols*panel, n_rows*(panel+0.3)))
    axes = np.atleast_2d(axes)
    for r, lname in enumerate(layer_names):
        act = acts[lname][0]                           # [channels, H, W]
        axes[r, 0].imshow(denormalize(image))
        axes[r, 0].set_title(f'{lname}\n(input)', loc='left', fontsize=7, pad=2)
        axes[r, 0].axis('off')
        for c in range(max_maps):
            ax = axes[r, c + 1]; ax.axis('off')
            if c >= act.shape[0]: continue
            m = act[c]
            m = (m - m.min()) / (m.max() - m.min() + 1e-8)
            ax.imshow(m, cmap='magma')
    fig.suptitle('Feature maps: bright cells = strong activation of that filter',
                 fontsize=11, y=1.02)
    plt.tight_layout(); plt.show()

# Grab one test image of a given class (normalised, ready for the model).
def pick_sample(loader, class_idx, max_batches=50):
    for images, labels in loader:
        hits = (labels == class_idx).nonzero(as_tuple=True)[0]
        if len(hits) > 0:
            return images[hits[0]]
    return images[0]

In [ ]:
# --- t-SNE: project penultimate features to 2-D ------------------------------------

# Run images through the model and collect the INPUT of the last Linear layer --
# i.e. the learned feature vector right before the classification decision.
# Works for ANY TorchVision CNN because we hook the last Linear generically.
@torch.no_grad()
def collect_features(model, loader, device, max_images=1000):
    parent, attr_name, last_linear = _find_last_linear(model)
    feats, labels = [], []
    handle = last_linear.register_forward_pre_hook(
        lambda module, args: feats.append(args[0].detach().cpu()))
    model.eval()
    seen = 0
    for images, ys in loader:
        model(images.to(device))
        labels.extend(ys.tolist())
        seen += images.size(0)
        if seen >= max_images:
            break
    handle.remove()
    X = torch.cat(feats)[:max_images].numpy()
    y = np.array(labels[:max_images])
    print(f'Collected features: {X.shape[0]} images x {X.shape[1]} dims')
    return X, y

# 2-D t-SNE scatter, coloured by true class.
def show_tsne(X, y, class_names, title='t-SNE of features'):
    from sklearn.manifold import TSNE
    n = len(y)
    emb = TSNE(n_components=2, perplexity=min(30, max(5, n // 20)),
               init='pca', learning_rate='auto', random_state=SEED
               ).fit_transform(X)
    plt.figure(figsize=(6.4, 5.2))
    cmap = plt.get_cmap('tab10' if len(class_names) <= 10 else 'tab20')
    for ci in np.unique(y):
        m = y == ci
        plt.scatter(emb[m, 0], emb[m, 1], s=8, color=cmap(ci % 20),
                    label=class_names[ci], alpha=0.75)
    plt.title(title, fontsize=10)
    plt.xticks([]); plt.yticks([])
    plt.legend(markerscale=2, fontsize=7, loc='center left',
               bbox_to_anchor=(1.0, 0.5))
    plt.tight_layout(); plt.show()

# Overlay one metric (default: test accuracy) across several experiments.
def plot_histories(histories, metric='test_acc', title=''):
    plt.figure(figsize=(7.2, 4.2))
    for name, h in histories.items():
        done = [e for e in h if e.get(metric) is not None]
        if not done: continue
        plt.plot([e['epoch'] for e in done], [e[metric] for e in done],
                 'o-', label=name)
    plt.xlabel('epoch'); plt.ylabel(metric.replace('_', ' '))
    plt.title(title or f'Comparison: {metric}', fontsize=10)
    plt.grid(alpha=0.3); plt.legend(fontsize=8)
    plt.tight_layout(); plt.show()

# Trainable-parameter count per strategy as a bar chart (random init, no download).
def plot_trainable_params(model_name='vgg16', num_classes=10):
    names, counts = [], []
    for s in ('head', 'frozen', 'unfreeze1', 'unfreeze2', 'unfreeze3', 'full'):
        m = build_model(model_name, num_classes, pretrained=False)
        apply_strategy(m, s, num_classes)
        names.append(STRATEGY_LABELS[s])
        counts.append(count_trainable(m) / 1e6)
    plt.figure(figsize=(7.6, 3.8))
    bars = plt.barh(names[::-1], counts[::-1], color='#3b6ea5')
    plt.xscale('log')
    plt.xlabel('trainable parameters (millions, log scale)')
    plt.title(f'Trainable parameters per strategy — {model_name}', fontsize=10)
    for b, v in zip(bars, counts[::-1]):
        plt.text(b.get_width()*1.15, b.get_y()+b.get_height()/2,
                 f'{v:,.2f}M', va='center', fontsize=8)
    plt.xlim(right=max(counts)*6)
    plt.tight_layout(); plt.show()
    print('Note the log scale: the same network, six wildly different learning budgets.')

### 3.8 Toolkit recap — the function reference card

| Function | One-line description |
|----------|----------------------|
| `load_data(name, image_size, batch_size, max_train, max_test)` | CIFAR-10/100 → ImageNet-normalised loaders |
| `build_model(name, num_classes, pretrained)` | any TorchVision CNN ± ImageNet weights |
| `replace_head(model, n, hidden=None)` | swap last Linear for a fresh n-class head |
| `freeze_all` / `unfreeze_all` | flip `requires_grad` everywhere |
| `freeze_all_conv_layers(model)` | freeze the conv backbone, keep dense layers trainable |
| `unfreeze_last_convs(model, n)` | release the last n conv layers |
| `apply_strategy(model, strategy, n)` | one call configures any of the 7 strategies |
| `report_trainable(model)` | print trainable/total parameter counts |
| `train_one_epoch` / `evaluate` | AMP-enabled loops, return (loss, accuracy) |
| `Experiment` / `init_experiment` / `train_experiment` | the uniform driver for all stages |
| `results_table()` | pandas DataFrame of every run so far |
| `show_kernels` / `show_feature_maps` / `show_tsne` | the three visualization families |
| `plot_history` / `plot_histories` / `plot_trainable_params` | curve & parameter charts |

> ✏️ **Question 3.3** — `train_one_epoch` calls `model.train()` at the start and `evaluate` calls
> `model.eval()`. Which VGG-16 components actually change behaviour between these two modes, and why
> would forgetting `model.eval()` inflate test accuracy *variance* (not just change it slightly)?
>
> <details><summary><b>Show answer</b></summary>
>
> VGG-16 contains **Dropout** layers (p = 0.5) in its classifier. In `train()` mode dropout randomly
> zeroes half the activations; in `eval()` mode it is disabled. Forgetting `model.eval()` means every
> test prediction is made through a *different random* subnetwork, so accuracy fluctuates run-to-run —
> noisy, unreproducible numbers instead of a stable estimate. (BatchNorm layers, which VGG-16 does not
> use but ResNet does, would misbehave even more: they would keep updating running statistics on test
> data.)
> </details>

---
The toolkit is complete. From here on, every experiment is a two-line cell: **`init_experiment` →
`train_experiment`**. Let the four stages begin.

## 4. Part 1 — Pre-trained model as-is (zero-shot)

**Research question for this stage:** how good is the ImageNet model on CIFAR *without any training at all*?

We test two versions of "as-is":

1. **Random head.** We replace the 1000-way head with a fresh 10-way (resp. 100-way) Linear and run the
   test set through. The convolutional backbone is fully pre-trained, but the last layer is random —
   accuracy should sit at random-guess level (10% / 1%). This isolates what the *head* contributes.
2. **Linear probe.** We freeze the whole backbone, record its 25,088-dimensional feature vector for a few
   thousand images, and fit a tiny *sklearn* logistic regression on top — no deep-learning training, no
   backpropagation through the CNN at all. If those features are as transferable as the theory claims,
   a *linear* read-out should already be far above chance.

Together these two numbers are the cleanest demonstration in the whole lecture: the random head says
"unadapted, the model knows nothing about CIFAR labels", while the probe says "but its *features* already
contain almost everything needed". Parts 2–7 then close the gap by actually training.

⏱ The probe forward-passes ~7,000 images per dataset (~2 min on T4) plus a fast sklearn fit.

In [ ]:
# --- Zero-shot experiment 1a: pre-trained VGG-16 on CIFAR-10, no training ----------
# init_experiment builds loaders + model + applies the 'zero_shot' strategy:
# the ImageNet head is swapped for a RANDOM 10-way head, nothing is frozen yet
# (irrelevant — we never take a gradient step in this stage).
exp_zero10 = init_experiment(model_name='vgg16', dataset_name='CIFAR10',
                             strategy='zero_shot')

# train_experiment recognises strategy='zero_shot' and only runs evaluation.
train_experiment(exp_zero10)

In [ ]:
# --- Zero-shot experiment 1b: same on CIFAR-100 (downloads CIFAR-100, ~170 MB) -----
# With a random 100-way head the random-guess level is 1%. The backbone is
# identical to the CIFAR-10 case — only the head width and dataset change,
# which is exactly the point of the unified API.
exp_zero100 = init_experiment(model_name='vgg16', dataset_name='CIFAR100',
                              strategy='zero_shot')

train_experiment(exp_zero100)

In [ ]:
# --- Linear probe: how good are the FROZEN features alone? --------------------------
# collect_features() hooks the input of the last Linear layer, i.e. we read out
# the 25,088-dim vector the backbone hands to the classifier. A sklearn logistic
# regression is then fitted on those vectors -- the cheapest possible 'read-out'
# training, and a lower bound on what fine-tuning can achieve.

def linear_probe(model, data, max_train=5000, max_test=2000):
    # Fit logistic regression on frozen backbone features; report test accuracy.
    from sklearn.linear_model import LogisticRegression
    Xtr, ytr = collect_features(model, data['train'], device, max_images=max_train)
    Xte, yte = collect_features(model, data['test'],  device, max_images=max_test)
    clf = LogisticRegression(max_iter=300, C=1.0)
    clf.fit(Xtr, ytr)
    acc = clf.score(Xte, yte)
    print(f'Linear probe accuracy: {acc:.4f}  ({acc*100:.1f}%) '
          f'[{clf.n_iter_[0]} optimiser iterations, zero CNN training]')
    return acc

print('=== CIFAR-10 ===')
probe_acc_10 = linear_probe(exp_zero10.model, exp_zero10.data)

print('\n=== CIFAR-100 ===')
probe_acc_100 = linear_probe(exp_zero100.model, exp_zero100.data)

**Expected lecture discussion.** The random head lands at ~10% on CIFAR-10 and ~1% on CIFAR-100 — exactly
chance, proof that accuracy does not come from the architecture but from the *learned weights*. The linear
probe, in contrast, typically reaches a solid two-digit percentage on CIFAR-10 from features that were
never trained on CIFAR — direct evidence that ImageNet features transfer. Also worth pointing out: the
probe used only 5,000 labelled images and a few thousand optimiser iterations on a *linear* model.

> ✏️ **Question 4.1** — The random head scored chance level even though 137 of 138 million parameters are
> ImageNet-trained. What does that tell us about where "knowledge" lives in a CNN?
>
> ✏️ **Question 4.2** — Why can a *linear* model read classes out of the frozen features at all? What
> property must those features have for a linear classifier to succeed?
>
> <details><summary><b>Show answers</b></summary>
>
> **4.1** Knowledge lives in the *match* between features and decision rule. The backbone computes rich
> features, but the last random layer destroys the mapping to the CIFAR label space — a model is only as
> good as its trained read-out. This motivates Part 2: train the read-out first.
>
> **4.2** The features must be (approximately) **linearly separable** with respect to the new classes.
> Deep ImageNet features famously are: each class occupies a compact, well-separated region of feature
> space, because "cat-ness" is already implicit in the detector activations. The probe is therefore the
> standard diagnostic for feature quality.
> </details>

---

## 5. Part 2 — Freezing layers

**Research question:** the features are good (§4) — how far do we get by training *only the classifier*
and never touching a single convolution weight?

We compare two freezing regimes, both driven by the same `apply_strategy` machinery:

* **`'head'`** — freeze *everything*, attach a brand-new small MLP head (4096→256→n, ≈1.05 M params)
  that reads the frozen 4096-dim output of the old dense layers. Fastest, smallest memory footprint.
* **`'frozen'`** — freeze all 13 conv layers, keep VGG's *dense* classifier (minus the replaced last
  layer) trainable: ≈119.6 M trainable parameters. This is "freeze the convolutions, retrain the dense
  part" — the classical transfer-learning recipe for VGG.

The bar chart below makes the parameter budgets tangible — note the **log scale**, or the head's bar would
be invisible next to the full model. Watch in the training curves how the two regimes trade off: capacity
and convergence speed versus over-fitting risk on only 50,000 images.

In [ ]:
# --- How big is each learning budget? Trainable params per strategy -----------------
# build_model(pretrained=False) -> no weight download, just the architecture,
# which is all we need to COUNT parameters. ~1 min on CPU for all six variants.
plot_trainable_params(model_name='vgg16', num_classes=10)

In [ ]:
# --- Experiment 2a: train ONLY the brand-new head (everything else frozen) ----------
# freeze_all + fresh MLP head -> ~1.05M trainable params, all in the new head.
exp_head = init_experiment(model_name='vgg16', dataset_name='CIFAR10',
                           strategy='head', epochs=CONFIG['epochs_frozen'])

train_experiment(exp_head)          # Adam @ lr=1e-3 -- fine for fresh weights

In [ ]:
# --- Experiment 2b: freeze conv backbone, train the dense classifier ----------------
# freeze_all_conv_layers + fresh last Linear -> ~119.6M trainable params.
# The ImageNet-taught dense layers now adapt to CIFAR's feature statistics.
exp_frozen = init_experiment(model_name='vgg16', dataset_name='CIFAR10',
                             strategy='frozen', epochs=CONFIG['epochs_frozen'])

train_experiment(exp_frozen)

In [ ]:
# --- Compare the two freezing regimes ----------------------------------------------
plot_histories({'new head only (1.05M trainable)': exp_head.history,
                'conv frozen, dense trained (119.6M)': exp_frozen.history},
               metric='test_acc',
               title='Part 2 — freezing strategies on CIFAR-10')

**Expected lecture discussion.** Both regimes massively beat the zero-shot baseline — training *something*
on top of good features is what unlocks accuracy. Typical observations worth naming in class: the small
head converges quickly but plateaus lower; the 119 M-parameter dense classifier fits faster and higher but
its train/test gap widens — capacity meets a 50k-image dataset. Neither touches the visual features:
everything so far is "adaptation on top of frozen perception".

> ✏️ **Question 5.1** — `exp_frozen` trains ≈119.6 M parameters on 50,000 images — a ratio of ~2,400
> parameters per training image. Why doesn't it simply memorise the training set and collapse to useless
> test accuracy?
>
> ✏️ **Question 5.2** — Both experiments freeze the convolutions. In which sense is `exp_head` the more
> "conservative" transfer, and when would you prefer it over `exp_frozen`?
>
> <details><summary><b>Show answers</b></summary>
>
> **5.1** Three brakes act together: (1) the *inputs* it classifies are frozen, ImageNet-grade features —
> the hard perceptual work is done, the dense layers only re-cut an already-good representation;
> (2) **Dropout (p=0.5)** inside the classifier injects noise every step; (3) **Adam with early-ish
> stopping** (we keep the epoch budget small) limits how far optimisation can go. Memorisation pressure
> exists — watch the train/test gap — but the frozen-feature input keeps it in check.
>
> **5.2** `exp_head` changes *fewer pre-trained decisions*: it trains only brand-new weights, so it can
> never damage the pre-trained representation. Prefer it when data is scarce, when the new task is close
> to ImageNet, when memory/time is tight, or as the safe first run of any transfer-learning project —
> exactly the workflow this notebook follows.
> </details>

---

## 6. Part 3 — Progressive unfreezing

**Research question:** the classifier is adapted — but the *features* were tuned for ImageNet. Which
convolutions are worth releasing for CIFAR, and how much does each release buy?

CNN features are hierarchical: block 1 holds generic edges, block 5 holds near-object-level detectors.
**Progressive unfreezing** releases them from the top down — first only the last conv layer, then the
last two, then the last three — re-assessing performance at each step:

* `unfreeze1` → `features.28` (block 5, conv 3) trainable
* `unfreeze2` → `features.26` + `features.28` (block 5, convs 2–3)
* `unfreeze3` → `features.24` + `features.26` + `features.28` (all of block 5)

Two things to watch. **Learning rate:** every unfreeze stage uses `3e-4` instead of `1e-3` — pre-trained
conv weights are tuned instruments, and a large step would knock them out of calibration. **Compute:**
backprop now flows into convolution layers, so epochs get noticeably slower than in Part 2. Each run gets
`CONFIG['epochs_unfreeze']` epochs; the comparison plot at the end stacks all runs against each other.

In [ ]:
# --- Experiment 3a: unfreeze the LAST conv layer only (features.28) -----------------
# 'frozen' setup + unfreeze_last_convs(model, 1): the dense classifier AND the
# deepest 512->512 convolution now receive gradients; everything else stays frozen.
exp_u1 = init_experiment(model_name='vgg16', dataset_name='CIFAR10',
                         strategy='unfreeze1', epochs=CONFIG['epochs_unfreeze'])

train_experiment(exp_u1)            # lr drops to 3e-4 -- gentle steps for pre-trained weights

In [ ]:
# --- Experiment 3b: unfreeze the last TWO conv layers (features.26 + features.28) ---
exp_u2 = init_experiment(model_name='vgg16', dataset_name='CIFAR10',
                         strategy='unfreeze2', epochs=CONFIG['epochs_unfreeze'])

train_experiment(exp_u2)

In [ ]:
# --- Experiment 3c: unfreeze the last THREE conv layers (all of block 5) ------------
exp_u3 = init_experiment(model_name='vgg16', dataset_name='CIFAR10',
                         strategy='unfreeze3', epochs=CONFIG['epochs_unfreeze'])

train_experiment(exp_u3)

In [ ]:
# --- Stack all four strategies so far ----------------------------------------------
plot_histories({'2a head only':            exp_head.history,
                '2b conv frozen':          exp_frozen.history,
                '3a last 1 conv':          exp_u1.history,
                '3b last 2 convs':         exp_u2.history,
                '3c last 3 convs':         exp_u3.history},
               metric='test_acc',
               title='Part 3 — progressive unfreezing on CIFAR-10')

**Expected lecture discussion.** Unfreezing the deepest layers usually buys a visible jump over
`exp_frozen`: the last block's detectors re-tune from ImageNet objects toward CIFAR's low-resolution
world. The gains from the second and third release are smaller — diminishing returns as we approach
layers whose features are already generic. Students should also notice the cost side of the ledger: each
release adds wall-clock time per epoch and, in the results table, trainable parameters.

> ✏️ **Question 6.1** — Why unfreeze from the *top* (output side) down, rather than the bottom up?
>
> ✏️ **Question 6.2** — `unfreeze3` trains block 5 with lr = 3e-4 while the head used 1e-3 in Part 2.
> What typically breaks if you fine-tune pre-trained convolutions at the head's higher rate?
>
> <details><summary><b>Show answers</b></summary>
>
> **6.1** Deeper layers encode task-specific abstractions; earlier layers encode near-universal edges and
> colours. Releasing the top lets the task-specific part adapt first while the generic foundation stays
> intact; it also keeps each experiment's trainable budget small so we can *attribute* gains to specific
> layers. Bottom-up unfreezing would perturb the shared foundation before the head has learned to use it.
>
> **6.2** Catastrophic forgetting / feature destruction: the large gradients from the fresh head flow
> into carefully calibrated ImageNet weights and shove them off their learned optimum — training loss
> falls while test accuracy stalls or drops (the classic fine-tuning instability). Lower rates — or
> per-layer rates — are the standard remedy.
> </details>

---

## 7. Part 4 — Full fine-tuning

**Research question:** what happens when *every* parameter is allowed to move?

`strategy='full'` unfreezes all 138 M parameters and re-creates the 10-way head. The whole network is now
trained end-to-end at **lr = 1e-4** — ten times gentler than the head-only stage, for exactly the reason
discussed in Question 6.2: with everything moving, a too-large step damages features everywhere at once.
Full fine-tuning is the strongest (and most expensive) transfer regime; it is the run to beat in the
results table.

In [ ]:
# --- Experiment 4: full fine-tuning -- all 138M parameters trainable ----------------
# unfreeze_all + fresh head; Adam @ lr=1e-4. The slowest stage per epoch, but
# every layer is now co-adapting to CIFAR.
exp_full = init_experiment(model_name='vgg16', dataset_name='CIFAR10',
                           strategy='full', epochs=CONFIG['epochs_full'])

train_experiment(exp_full)

**Expected lecture discussion.** Full fine-tuning typically reaches the best test accuracy of all stages —
but compare *marginal gain per resource*: how many extra epochs, how much extra wall-clock time and GPU
memory did it cost over `unfreeze3`? That trade-off, not the raw leaderboard, is the engineering
take-away. In the training curves, also watch for the signature of fine-tuning done right: steady test
improvement *without* the training loss collapsing toward zero (which would signal forgetting).

> ✏️ **Question 7.1** — Full fine-tuning won on accuracy. Name two realistic scenarios where you would
> still *not* deploy it.
>
> <details><summary><b>Show answer</b></summary>
>
> (1) **Small data + distribution shift:** with a few thousand images, 138 M movable weights over-fit or
> drift; frozen features + small head generalise better. (2) **Deployment economics:** full fine-tuning
> costs the most GPU time to train *and* you must ship/store the whole network per task, while a frozen
> backbone lets you share one feature extractor across many cheap heads (and skip backbone gradients in
> training entirely). Edge devices and multi-task services often prefer the frozen-backbone arrangement.
> </details>

## 8. Results summary

Every `train_experiment` call since §4 appended one row to the global `RESULTS` book. Time to read it.
The table reports, for each run: the strategy, how many parameters were actually trainable, the training
budget, and test accuracy. Below it, all accuracy curves are overlaid in a single figure — the whole
lecture's story in one picture: *chance → probe → head → frozen → progressively unfrozen → full*.

In [ ]:
# --- The complete results book ------------------------------------------------------
df = results_table()
print(f'{len(df)} experiments completed\n')
display(df)

# Same table, sorted by accuracy, for quick reading
print('\nSorted by best test accuracy:')
display(df.sort_values('best_test_acc', ascending=False).reset_index(drop=True))

In [ ]:
# --- One figure: test accuracy across ALL stages ------------------------------------
plot_histories({
    'zero-shot (chance level)':      exp_zero10.history,
    '2a new head only':              exp_head.history,
    '2b conv frozen, dense trained': exp_frozen.history,
    '3a last 1 conv unfrozen':       exp_u1.history,
    '3b last 2 convs unfrozen':      exp_u2.history,
    '3c last 3 convs unfrozen':      exp_u3.history,
    '4 full fine-tuning':            exp_full.history,
}, metric='test_acc', title='The whole lecture in one plot — CIFAR-10 test accuracy')

**How to read the results in class.** Three axes are worth explicit discussion. *Features vs read-out:*
the probe (§4) already extracted real accuracy from frozen features with a linear model; training the
head (§5) converted that potential into end-to-end accuracy. *Depth of adaptation:* each unfreezing step
in §6 bought progress with a growing parameter bill — the table's `trainable_M` column makes the price
visible. *Diminishing returns:* the gap between `unfreeze3` and `full` is usually the smallest jump in
the whole table, while being the most expensive run — the empirical argument for the progressive recipe
of Part 3. Encourage students to frame every future transfer-learning decision as choosing a point on
these axes rather than defaulting to "train everything".

---

## 9. Inside the model: kernels, feature maps & t-SNE

Accuracy tables say *how well* the model works; this section opens it up and shows *what it learned*.
We use the fine-tuned network from Part 4 (`exp_full.model`) — but every cell below works verbatim on any
model from the toolkit, including the pristine pre-trained one (`exp_zero10.model`), and comparing the two
is a recommended exercise.

**VGG-16 layer map.** The convolutional backbone (`model.features`) is a flat `nn.Sequential` — layer
indices are integer positions in it. Our four visualisation targets and the three unfreezing targets all
live here:

| index | block | role | lecture relevance |
|-------|-------|------|-------------------|
| `features.0`  | block 1, conv 1 | 3→64, 3×3 | **1st conv layer** — RGB edge/colour detectors |
| `features.2`  | block 1, conv 2 | 64→64, 3×3 | **2nd conv layer** — combinations of edges |
| `features.5`  | block 2, conv 1 | 64→128, 3×3 | **3rd conv layer** *and* **1st conv of block 2** (the same layer!) |
| `features.10` | block 3, conv 1 | 128→256, 3×3 | deeper texture/part detectors for contrast |
| `features.28` | block 5, conv 3 | 512→512, 3×3 | **last conv layer** — released first in §6 |

> 💡 A nice VGG coincidence: the **3rd convolution of the network is also the 1st convolution of block 2**
> (`features.5`), because block 1 contains exactly two conv layers. When the lecture plan asks for both,
> it is one tensor — we additionally show `features.10` so four *distinct* layers are on display.

In [ ]:
# --- Inventory: name and size of every conv layer in the model ----------------------
# This prints the exact names our visualization / unfreezing functions accept.
print(f'Convolutional layers of {exp_full.model_name}:')
for name, m in get_conv_names(exp_full.model):
    n_params = sum(p.numel() for p in m.parameters())
    print(f'  {name:<14} Conv2d({m.in_channels:>3} -> {m.out_channels:>3}, '
          f'{m.kernel_size[0]}x{m.kernel_size[0]})  {n_params/1e6:6.2f}M params')

In [ ]:
# --- Kernel grids: the 1st, 2nd, 3rd, deeper, and last conv layers -------------------
# Row per layer, up to 16 filters per row. features.0 filters are RGB patches;
# for deeper layers (many input channels) we draw each filter's first channel.
show_kernels(exp_full.model,
             layer_names=['features.0', 'features.2', 'features.5',
                          'features.10', 'features.28'],
             max_filters=16)

**Reading the kernel grids.** `features.0` shows the famous elementary detectors: red/green opposing
pairs, blue/yellow pairs, oriented luminance edges — learned, not hand-crafted, and remarkably consistent
across independently trained networks. `features.2` already looks abstract: each of its 64 filters spans
64 input channels, so a single 3×3 grayscale patch is only a *slice* through a 64-channel operator —
row-to-row structure and smoothness matter more than any one patch. From `features.5` onward patterns are
effectively decorrelated from human-interpretable edge detectors: depth buys abstraction at the cost of
direct interpretability. When comparing with `exp_zero10.model`, expect the deep-layer kernels to differ
most — those are the weights fine-tuning touched (block 5) and re-purposed for CIFAR.

Next we turn the question around: not *what does each filter look like*, but *what does it fire on*.

In [ ]:
# --- Feature maps: activations of the same five layers for ONE image -----------------
# Pick a test image (class 'cat' = index 3 of CIFAR-10), then record each layer's
# output while the image flows through the network. Each row below: input image
# followed by that layer's first 16 feature maps (bright = strong activation).
sample = pick_sample(exp_full.data['test'], class_idx=3)
print('Sample chosen from the CIFAR-10 test set, true class "cat"')

show_feature_maps(exp_full.model, sample,
                  layer_names=['features.0', 'features.2', 'features.5',
                               'features.10', 'features.28'],
                  max_maps=16)

**Reading the feature maps.** Follow a bright region diagonally: in `features.0` only the filters matching
the sample's *edges and colours* are lit, and their maps still look like the input (a cat-shaped mask).
By `features.5` activations respond to contours and fur-like textures; at `features.10` they light up on
parts (ears, face region) rather than pixels; and the `features.28` maps are coarse 7×7 summaries —
"cat evidence somewhere in this region". Two structural facts to point out: spatial size shrinks block by
block (each MaxPool halves it: 224→112→56→28→14→7) while the *number* of channels grows (64→128→256→512)
— resolution is traded for abstraction. And a filter that stays dark did not find its trigger in this
image: activation is selective, which is exactly what makes the features informative.

> ✏️ **Question 9.1** — The `features.28` maps are 7×7 while the input is 224×224. Which layers caused
> each reduction, and what replaces lost spatial precision?
>
> <details><summary><b>Show answer</b></summary>
>
> Five `MaxPool2d(2)` layers (after each block) halve resolution five times: 224→112→56→28→14→7.
> What replaces spatial precision is **channel abstraction**: 64 → 128 → 256 → 512 channels, each a
> learned "concept detector" pooling over a large receptive field. The classifier ultimately reads the
> 7×7×512 = 25,088 numbers as a flat feature vector — position information has been deliberately
> squeezed out in favour of "which detectors fired, roughly where".
> </details>

In [ ]:
# --- t-SNE: does fine-tuning reorganise the feature space? ---------------------------
# Collect the 25,088-dim penultimate features of the same 1,000 test images from
# TWO networks: the pristine pre-trained backbone (exp_zero10.model) and the
# fully fine-tuned one (exp_full.model). Then project both to 2-D with t-SNE.
print('Extracting features BEFORE fine-tuning (pre-trained backbone)...')
X_before, y_before = collect_features(exp_zero10.model, exp_zero10.data['test'],
                                      device, max_images=1000)

print('Extracting features AFTER full fine-tuning...')
X_after, y_after = collect_features(exp_full.model, exp_full.data['test'],
                                    device, max_images=1000)

show_tsne(X_before, y_before, exp_zero10.data['classes'],
          title='Before fine-tuning — pre-trained features')
show_tsne(X_after, y_after, exp_full.data['classes'],
          title='After full fine-tuning — adapted features')

**Reading the t-SNE pair.** Both plots use the *same 1,000 test images* and the *same* projection method —
only the network differs. Before fine-tuning the ten classes form loose, partly overlapping islands: the
ImageNet features are good but their geometry was shaped for 1,000 ImageNet classes, not these ten.
After fine-tuning the islands contract and separate — the network has *re-carved feature space* around the
CIFAR class boundaries, which is the geometric signature of the accuracy gains in §8. (t-SNE caveat worth
teaching: inter-cluster *distances* are not meaningful, only within-cluster structure and separability.)

> ✏️ **Question 9.2** — Suppose you repeated this t-SNE with the *randomly initialised* (never trained)
> network. What would the plot look like, and why?
>
> <details><summary><b>Show answer</b></summary>
>
> Class colours would be scattered nearly uniformly — no island structure at all. Random convolutions
> produce generic, essentially texture-hash features that carry almost no class information (this is the
> feature-space counterpart of the ~10% random-head accuracy of §4). Structure in feature space is
> *learned*, and both the zero-shot islands and the fine-tuned separation are artefacts of training —
> just at different scopes (ImageNet vs CIFAR).
> </details>

---

## 10. Summary & further reading

**What the notebook established, stage by stage.** A pre-trained CNN transferred to CIFAR *without any
training* is useless through a random head but far above chance through a linear probe — the features
transfer, the read-out does not. Training a fresh head on frozen features converts that potential into
real accuracy at negligible cost. Releasing the last 1–3 convolutions (with a reduced learning rate) buys
further gains in measured, attributable steps. Full fine-tuning gives the best accuracy at the highest
price. Along the way, kernel grids showed *what the filters are*, feature maps showed *what they fire on*,
and t-SNE showed *how fine-tuning reorganises feature space*. And the entire study ran on one uniform
API — `init_experiment → train_experiment → results_table` — in which swapping the model or dataset is a
two-string change: the same cells run `resnet18`, `mobilenet_v2`, or CIFAR-100 unmodified.

**The decision recipe this lecture leaves you with:**

1. Start with a **linear probe** on frozen features to measure transferable quality (§4).
2. Train the **new head** first — cheapest, safest step (§5).
3. **Unfreeze progressively** from the top with a *lower* learning rate, re-evaluating at each step (§6).
4. Go **full fine-tuning** only if the data and the accuracy budget justify 138 M trainable parameters (§7).
5. **Look inside** before, during and after: kernels, feature maps and t-SNE catch what accuracy hides (§9).

**Further reading**

* Simonyan & Zisserman, *Very Deep Convolutional Networks for Large-Scale Image Recognition* (2014) — the VGG paper; §3 of the paper maps 1:1 to the layer table above.
* Yosinski et al., *How transferable are features in deep neural networks?* (NeurIPS 2014) — the classic progressive-unfreezing experiment; Part 3 of this notebook is its classroom version.
* Zeiler & Fergus, *Visualizing and Understanding Convolutional Networks* (ECCV 2014) — deconvolutional feature-map visualization, the ancestor of §9.
* CS231n notes, *Transfer Learning* — Stanford's practical recipes; matches the decision recipe above.
* PyTorch documentation: *Optimizing Model Parameters*, and the official *Transfer Learning for Computer Vision* tutorial — the `requires_grad` mechanics used throughout.

**Try next (5-minute experiments on the same API):** set `CONFIG['model_name']='resnet18'` and rerun §5–§8
— how do the parameter bars and curves change? Set `CONFIG['dataset_name']='CIFAR100'` and compare the
probe and full fine-tuning gaps. Or add `'unfreeze5'` to `STRATEGIES` — does releasing two more layers
still pay off? Each question is one config string away.